# DT401 Week 2 — SQL Part 6: Advanced Practice

## Learning Objectives

By the end of this notebook, you will be able to:

- Construct multi-table SQL queries spanning four to six tables
- Apply `CASE` statements to create conditional classifications within a query
- Use string functions (`CONCAT`, `SUBSTRING`) and date functions (`strftime`) to transform column values
- Use self-joins to compare rows within the same table
- Approach an open-ended analytical question by building a series of targeted queries

---

## Context Recap

You have now covered the core SQL toolkit: selecting, filtering, sorting, aggregating, joining, and creating tables. This notebook brings those skills together in questions that require you to combine several techniques at once. A few questions also introduce new functions — `CASE`, `CONCAT`, and `strftime` — which extend what you can do without leaving SQL.

These questions are designed to challenge you. Working carefully through a solution you did not arrive at independently is just as valuable as solving it from scratch — the goal is to understand the approach, not simply to produce an answer.

**Prerequisites:** Complete Notebooks 1–5 before starting this notebook.

Run the setup cell before starting.

In [ ]:
# Setup: load the SQL extension and connect to the pagila database
%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False
%config SqlMagic.autopolars = False
%config SqlMagic.displaycon = False

%sql sqlite:///data/pagila.db

> **Run this cell before starting.** If your kernel restarts at any point, run it again before executing any SQL cells.

---

## Stretch Questions

These questions combine multiple SQL concepts and, in some cases, introduce functions you have not seen before. Brief introductions are provided where a new concept appears. Read them before attempting the question — they give you enough to get started, but you will still need to think carefully about how the pieces fit together.

Work through them in order if you can: the earlier questions build techniques that later questions rely on.

---

### Stretch I

> **Question**
>
> What is the average film length by genre?
>
> <details>
> <summary>💡 Hint</summary>
>
> You will need to join `film`, `film_category`, and `category`.
>
> </details>

In [ ]:
%%sql


<details>
<summary>Solution — Stretch I</summary>

```sql
SELECT
    c.name AS category,
    ROUND(AVG(f.length), 2) AS avg_length
FROM film f
INNER JOIN film_category fc ON f.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
GROUP BY c.name
ORDER BY avg_length DESC;
```

Sports has the longest average film length. This query requires two joins: first from `film` to `film_category` (the junction table linking films to genres), then from `film_category` to `category` to retrieve the genre name.

</details>

---

### Stretch II

> **Question**
>
> What is the most profitable genre? You will need to trace payments through rentals, inventory, films, and categories — six tables in total.

In [ ]:
%%sql


<details>
<summary>Solution — Stretch II</summary>

```sql
SELECT
    c.name AS category,
    SUM(p.amount) AS total_sales
FROM film f
INNER JOIN inventory i ON f.film_id = i.film_id
INNER JOIN rental r ON i.inventory_id = r.inventory_id
INNER JOIN payment p ON r.rental_id = p.rental_id
INNER JOIN film_category fc ON f.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
GROUP BY c.name
ORDER BY total_sales DESC;
```

Sports is the most profitable genre. The chain of joins traces the revenue path: a film sits in inventory, inventory items are rented, and rentals generate payments. The `film_category` and `category` joins add the genre label.

</details>

---

### Stretch III

> **Question**
>
> How many different genres has the average customer rented? Use a subquery.

In [ ]:
%%sql


<details>
<summary>Solution — Stretch III</summary>

```sql
SELECT ROUND(AVG(distinct_categories), 2) AS avg_distinct_categories
FROM (
    SELECT
        p.customer_id,
        COUNT(DISTINCT c.name) AS distinct_categories
    FROM film f
    INNER JOIN inventory i ON f.film_id = i.film_id
    INNER JOIN rental r ON i.inventory_id = r.inventory_id
    INNER JOIN payment p ON r.rental_id = p.rental_id
    INNER JOIN film_category fc ON f.film_id = fc.film_id
    INNER JOIN category c ON fc.category_id = c.category_id
    GROUP BY p.customer_id
) x;
```

The average customer has rented from 12.92 distinct genres. The inner query calculates a distinct genre count per customer; the outer query then averages those counts across all customers.

</details>

---

### Stretch IV

> **Question**
>
> Who is the most profitable actor — the actor whose films have generated the most total revenue?

In [ ]:
%%sql


<details>
<summary>Solution — Stretch IV</summary>

```sql
SELECT
    a.first_name,
    a.last_name,
    SUM(p.amount) AS total_sales
FROM actor a
INNER JOIN film_actor fa ON a.actor_id = fa.actor_id
INNER JOIN inventory i ON fa.film_id = i.film_id
INNER JOIN rental r ON i.inventory_id = r.inventory_id
INNER JOIN payment p ON r.rental_id = p.rental_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY total_sales DESC;
```

Gina Degeneres is the most profitable actor. Note that `GROUP BY` includes `actor_id` as well as the name columns — this ensures that two actors who happen to share a name are not merged into a single row.

</details>

---

### Stretch V — Self-join and date arithmetic

Stretch V introduces a **self-join**: joining a table to itself. This is useful when you want to compare rows within the same table — here, finding pairs of payments made by the same customer so you can measure the gap between them.

> **SQLite note:** SQLite stores timestamps as text strings rather than native date values, so standard timestamp subtraction does not work as it would in PostgreSQL or SQL Server. The solution below uses `julianday()`, which converts a date string to a fractional day number, making arithmetic possible.

> **Question**
>
> What is the longest gap between two consecutive payments by the same customer?
>
> <details>
> <summary>💡 Hint</summary>
>
> Join the `payment` table to itself on `customer_id`, using `a.payment_date > b.payment_date` to find pairs where one payment follows another. Use `MAX(b.payment_date)` to find the most recent earlier payment for each row. Store intermediate results in a temporary table.
>
> </details>

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution — Stretch V</summary>

```sql
CREATE TABLE subsequent_dates AS
SELECT
    a.payment_date,
    MAX(b.payment_date) AS previous_payment_date
FROM payment a
LEFT JOIN payment b
    ON a.payment_date > b.payment_date
    AND a.customer_id = b.customer_id
GROUP BY a.payment_date;
```

```sql
SELECT MAX(julianday(payment_date) - julianday(previous_payment_date)) AS max_gap_days
FROM subsequent_dates;
```

```sql
DROP TABLE subsequent_dates;
```

The longest gap is approximately 46.7 days. `julianday()` converts each timestamp to a decimal day count, making subtraction straightforward. The `LEFT JOIN` ensures that even the very first payment (which has no earlier payment to compare against) is retained in the table — it will have a `NULL` in `previous_payment_date`, which `MAX()` ignores.

</details>

---

### Stretch VI — `CASE` statements

SQL's `CASE` statement works like an if/else in other programming languages. It evaluates conditions in order, row by row, and returns the value for the first condition that matches. The syntax is:

```sql
CASE
    WHEN condition_1 THEN value_1
    WHEN condition_2 THEN value_2
    ELSE default_value
END
```

The `ELSE` clause captures any row that does not match the earlier conditions. If you omit it, unmatched rows return `NULL`.

> **Question**
>
> Create a copy of the `payment` table with a new column classifying each payment as:
> - `low` if the amount is less than 3
> - `medium` if between 3 and 4 (inclusive)
> - `high` if greater than 4

In [ ]:
%%sql


<details>
<summary>Solution — Stretch VI</summary>

```sql
SELECT *,
    (CASE
        WHEN amount < 3 THEN 'low'
        WHEN amount > 4 THEN 'high'
        ELSE 'medium'
    END) AS amount_class
FROM payment;
```

The conditions are evaluated in order: amounts below 3 are labelled `low`, amounts above 4 are labelled `high`, and everything else (3 to 4 inclusive) falls into the `ELSE` clause and is labelled `medium`. Notice that the `CASE` expression sits inside the `SELECT` list — it creates a new derived column without modifying the underlying table.

</details>

---

### Stretch VII — String functions

SQL provides functions for manipulating text values. Two of the most useful are:

- **`SUBSTRING(string, start, length)`** — extracts a portion of a string, starting at position `start` (1-indexed) for `length` characters. `SUBSTRING('KARL', 1, 3)` returns `'KAR'`.
- **`CONCAT(str1, str2)`** — joins two strings together. `CONCAT('KAR', 'SEA')` returns `'KARSEA'`.

> **SQLite note:** Some SQLite builds support `CONCAT()`, but the `||` operator is more reliably supported for concatenation in SQLite: `'KAR' || 'SEA'` returns `'KARSEA'`. The solution below uses `CONCAT()` — if it produces an error in your environment, replace `CONCAT(x, y)` with `x || y`.

> **Question**
>
> Create a copy of the `customer` table with a new `username` column. The username should take the first three letters of `first_name` and the first three letters of `last_name`, joined together. For example, `KARL SEAL` becomes `KARSEA`.

In [ ]:
%%sql


<details>
<summary>Solution — Stretch VII</summary>

```sql
SELECT *,
    CONCAT(SUBSTRING(first_name, 1, 3), SUBSTRING(last_name, 1, 3)) AS username
FROM customer;
```

`SUBSTRING(first_name, 1, 3)` extracts the first three characters of the first name; the same pattern applied to `last_name` extracts the first three characters of the surname. `CONCAT()` joins the two results into a single string. If your SQLite environment does not support `CONCAT()`, the equivalent is `SUBSTRING(first_name, 1, 3) || SUBSTRING(last_name, 1, 3)`.

</details>

---

### Stretch VIII — Date functions

`strftime(format, column)` extracts parts of a date in SQLite. For example, `strftime('%m', payment_date)` returns the month as a two-character string (`'04'` for April). It can be wrapped with `CAST(... AS INTEGER)` to convert the string to a number.

Common format codes:

| Code | Returns |
|:-----|:--------|
| `%Y` | Four-digit year |
| `%m` | Month (01–12) |
| `%d` | Day of month (01–31) |
| `%w` | Day of week (0 = Sunday, 1 = Monday, …, 6 = Saturday) |

> **SQLite-specific:** `strftime()` is a SQLite function. Other databases use different syntax — PostgreSQL uses `EXTRACT(MONTH FROM payment_date)`, SQL Server uses `MONTH(payment_date)`. If you move to a different database system, you will need to adapt this pattern.

> **Question**
>
> Which month sees the most total sales? Which day of the month? Which day of the week?

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution — Stretch VIII</summary>

```sql
SELECT
    CAST(strftime('%m', payment_date) AS INTEGER) AS payment_month,
    SUM(amount) AS total
FROM payment
GROUP BY strftime('%m', payment_date)
ORDER BY total DESC;
```

April (month 4) sees the most total sales.

```sql
SELECT
    CAST(strftime('%d', payment_date) AS INTEGER) AS payment_day,
    SUM(amount) AS total
FROM payment
GROUP BY strftime('%d', payment_date)
ORDER BY total DESC;
```

The 30th of the month sees the most sales.

```sql
SELECT
    CAST(strftime('%w', payment_date) AS INTEGER) AS day_of_week,
    SUM(amount) AS total
FROM payment
GROUP BY strftime('%w', payment_date)
ORDER BY total DESC;
```

Monday sees the most sales (`day_of_week` = 1; recall that 0 = Sunday). Note that `GROUP BY` uses the raw `strftime()` expression while `SELECT` uses `CAST(... AS INTEGER)` — this avoids sorting problems that can arise when grouping on an already-cast integer.

</details>

---

### Stretch IX — Ranking with a self-join

Window functions such as `RANK()` and `ROW_NUMBER()` are the standard way to assign ranks in production databases — PostgreSQL, SQL Server, and MySQL all support them. SQLite does not support window functions in older versions, so this question shows an alternative approach using a self-join.

The principle: a film's rank equals 1 plus the number of other films with strictly higher sales than it. You can calculate this by joining the sales summary table to itself, counting how many rows in the second copy have a higher sales value than the row in the first copy.

> **Question**
>
> Create a summary of total film sales and add a rank column based on which film generated the most revenue. The top-selling film should have rank 1.
>
> <details>
> <summary>💡 Hint</summary>
>
> First create a temporary table storing each `film_id` and its total sales. Then self-join that table: for each film `a`, count the number of films `b` where `b.sales > a.sales`. A film's rank is that count plus 1.
>
> </details>

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution — Stretch IX</summary>

```sql
CREATE TABLE film_sum AS
SELECT
    f.film_id,
    SUM(p.amount) AS sales
FROM film f
INNER JOIN inventory i ON f.film_id = i.film_id
INNER JOIN rental r ON i.inventory_id = r.inventory_id
INNER JOIN payment p ON r.rental_id = p.rental_id
GROUP BY f.film_id;
```

```sql
SELECT
    a.film_id,
    a.sales,
    COUNT(b.film_id) + 1 AS rank
FROM film_sum a
LEFT JOIN film_sum b ON a.sales < b.sales
GROUP BY a.film_id, a.sales
ORDER BY rank;
```

```sql
DROP TABLE film_sum;
```

The `LEFT JOIN` connects each film in `a` to all films in `b` that have strictly higher sales. `COUNT(b.film_id)` counts those higher-ranked films; adding 1 gives the film's rank. Films with equal sales receive the same rank (a dense rank is not produced here — this matches standard competition ranking behaviour). In a PostgreSQL or SQL Server environment, `RANK() OVER (ORDER BY SUM(p.amount) DESC)` would produce the same result more concisely.

</details>

---

### Stretch X — Complex analysis

> **Question**
>
> What is the most profitable combination of two actors who appear together in the same film?
>
> <details>
> <summary>💡 Hint</summary>
>
> Self-join `film_actor` to create all pairs of actors who share a `film_id`. Use `a.actor_id > b.actor_id` to ensure each pair appears only once (otherwise you would count Nick Wahlberg + Ed Chase and Ed Chase + Nick Wahlberg as two separate pairs). Then join the pairs through to payments.
>
> </details>

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution — Stretch X</summary>

```sql
CREATE TABLE actor_combos AS
SELECT
    a.film_id,
    a1.actor_id AS actor_id_1,
    a1.first_name AS first_name_1,
    a1.last_name AS last_name_1,
    a2.actor_id AS actor_id_2,
    a2.first_name AS first_name_2,
    a2.last_name AS last_name_2
FROM film_actor a
INNER JOIN film_actor b
    ON a.film_id = b.film_id AND a.actor_id > b.actor_id
INNER JOIN actor a1 ON a.actor_id = a1.actor_id
INNER JOIN actor a2 ON b.actor_id = a2.actor_id;
```

```sql
SELECT
    ac.first_name_1, ac.last_name_1,
    ac.first_name_2, ac.last_name_2,
    SUM(p.amount) AS total_sales
FROM actor_combos ac
INNER JOIN inventory i ON ac.film_id = i.film_id
INNER JOIN rental r ON i.inventory_id = r.inventory_id
INNER JOIN payment p ON r.rental_id = p.rental_id
GROUP BY
    ac.actor_id_1, ac.first_name_1, ac.last_name_1,
    ac.actor_id_2, ac.first_name_2, ac.last_name_2
ORDER BY total_sales DESC;
```

```sql
DROP TABLE actor_combos;
```

Henry Berry and Julia McQueen are the most profitable pairing. The condition `a.actor_id > b.actor_id` is the key to avoiding duplicates: it ensures that for any two actors, only the combination where the higher `actor_id` appears in `a` is retained. Without this condition, every pair would appear twice and revenue would be double-counted.

</details>

---

### Stretch XI — Open-ended analysis

This question has no single correct answer. The goal is to practise building an analytical argument from multiple queries — a skill that transfers directly to real-world data work.

> **Question**
>
> You need to recommend a film to **Karl Seal**. Which film would you recommend, and why?
>
> Use multiple queries to build a profile of Karl's viewing habits: his favourite genres, most-watched actors, and typical film length. Then find a film he has not yet rented that fits his profile.
>
> There is no single right answer — what matters is that your recommendation is supported by evidence from the data.

In [ ]:
%%sql


In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>One approach — Stretch XI</summary>

Start by identifying which films Karl has already seen:

```sql
CREATE TABLE karl_films AS
SELECT DISTINCT f.film_id
FROM film f
INNER JOIN inventory i ON f.film_id = i.film_id
INNER JOIN rental r ON i.inventory_id = r.inventory_id
INNER JOIN payment p ON r.rental_id = p.rental_id
INNER JOIN customer c ON p.customer_id = c.customer_id
WHERE c.first_name = 'KARL' AND c.last_name = 'SEAL';
```

Then find his most-rented genre from those films:

```sql
SELECT category_id, COUNT(*) AS freq
FROM film_category
WHERE film_id IN (SELECT film_id FROM karl_films)
GROUP BY category_id
ORDER BY freq DESC;
```

And his most-watched actors:

```sql
SELECT actor_id, COUNT(*) AS freq
FROM film_actor
WHERE film_id IN (SELECT film_id FROM karl_films)
GROUP BY actor_id
ORDER BY freq DESC;
```

```sql
DROP TABLE karl_films;
```

Karl's most-rented genre is **Animation** (category_id 2) and his most-watched actor is **JOHNNY LOLLOBRIGIDA** (actor_id 5). One well-supported recommendation is **Sunrise League** — an animation starring his favourite actor, at 135 minutes it also fits Karl's preference for longer films. Your recommendation may differ if you weight the evidence differently or add additional criteria, such as average rating or rental price.

</details>

---

## Summary

In this notebook you applied the full range of SQL skills from Notebooks 1–5 to multi-table problems, and extended your toolkit with four new techniques.

| Keyword / Function | Purpose |
|:-------------------|:--------|
| `CASE WHEN ... THEN ... ELSE ... END` | Conditional logic — returns different values depending on which condition matches |
| `CONCAT(str1, str2)` | Joins two strings together (use `\|\|` as an alternative in SQLite) |
| `SUBSTRING(string, start, length)` | Extracts a portion of a string |
| `strftime(format, col)` | Extracts year, month, day, or day-of-week from a date (SQLite-specific) |
| `julianday(date)` | Converts a date string to a decimal day number, enabling date arithmetic in SQLite |
| Self-join | Joins a table to itself to compare rows within the same table |

**Next:** [Notebook 7 — SQL Interactions in Python](7_sql_interactions.ipynb)